# PWC-Net — Confusion Matrix · ROC (200개) · **로컬 노트북**

GPU / S3 없이 **PC에서 이 노트북만** 실행합니다.

## 준비
1. JSON 2개는 이미 `ai/docs/notebooks/data/`에 있어야 합니다
2. **Kernel → Restart Kernel** 후 **셀 3부터 Run All** (또는 전체 Run All)
3. 셀 3이 `matplotlib` 등 없으면 자동 설치합니다

| profile | 파일 |
|---------|------|
| ffpp_vox | `data/pwcnet_ffpp_vox_benchmark_report.json` |
| celebdf | `data/pwcnet_celebdf_benchmark_report.json` |

프로필별 threshold는 **셀 3** `THRESHOLDS`에서 조정합니다 (기본: celebdf=0.19, ffpp_vox=0.5).

근거: [PWCNET_THRESHOLD_ANALYSIS_REPORT.md](../../PWCNET_THRESHOLD_ANALYSIS_REPORT.md)


출력 PNG: `output/pwcnet-cm/` (`cm_pwcnet_*_*.png`, `roc_pwcnet_*_*.png`)

팀 CM 표기: Xception/EfficientNet 노트북과 동일 (Positive = fake)


In [ ]:
# (선택) 아래 셀 3이 패키지를 자동 설치합니다. 이 셀은 건너뛰어도 됩니다.
# %pip install matplotlib seaborn scikit-learn numpy pandas

## JSON 다운로드 (최초 1회)

PowerShell에서 실행 (비밀번호 입력 필요). `DATA_DIR`는 아래 경로입니다.

```
c:\sw_study\finalpjt\ai\docs\notebooks\data
```

```powershell
$DATA = "c:\sw_study\finalpjt\ai\docs\notebooks\data"
New-Item -ItemType Directory -Force -Path $DATA | Out-Null

scp sk4team@58.127.241.84:~/forenShield-ai/results/infer/pwcnet-ffpp_vox-20260622-041348/benchmark_report.json "$DATA\pwcnet_ffpp_vox_benchmark_report.json"

scp sk4team@58.127.241.84:~/forenShield-ai/results/infer/pwcnet-celebdf-20260622-041153/benchmark_report.json "$DATA\pwcnet_celebdf_benchmark_report.json"
```

S3에서 받는 경우 (AWS 프로필 `forenshield` 등):
- `cases/test/video-benchmark-datasets/PWC-Net/ffpp_vox/benchmark_report.json`
- `cases/test/video-benchmark-datasets/PWC-Net/celebdf/benchmark_report.json`

In [ ]:
import json
import subprocess
import sys
from pathlib import Path


def _ensure_pkg(import_name: str, pip_name: str | None = None) -> None:
    pip_name = pip_name or import_name
    try:
        __import__(import_name)
    except ImportError:
        print(f"Installing {pip_name} into {sys.executable} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


for _mod, _pip in [
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("sklearn", "scikit-learn"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
]:
    _ensure_pkg(_mod, _pip)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib as mpl
import matplotlib.font_manager as fm

print("packages OK")


def _setup_korean_font() -> str:
    # CM 축 라벨(가짜/진짜) 한글이 깨지지 않게 폰트 설정
    candidates = [
        "Malgun Gothic",
        "NanumGothic",
        "Nanum Gothic",
        "AppleGothic",
        "Noto Sans CJK KR",
        "Noto Sans KR",
    ]
    available = {f.name for f in fm.fontManager.ttflist}
    for name in candidates:
        if name in available:
            mpl.rcParams["font.family"] = name
            mpl.rcParams["axes.unicode_minus"] = False
            return name
    mpl.rcParams["axes.unicode_minus"] = False
    return "default (Korean font not found — install Malgun Gothic or NanumGothic)"


_kr_font = _setup_korean_font()
print(f"matplotlib font: {_kr_font}")

NOTEBOOK_DIR = Path(r"c:\sw_study\finalpjt\ai\docs\notebooks")
DATA_DIR = NOTEBOOK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_FILES = {
    "ffpp_vox": DATA_DIR / "pwcnet_ffpp_vox_benchmark_report.json",
    "celebdf": DATA_DIR / "pwcnet_celebdf_benchmark_report.json",
}

# threshold sweep 결과 기반 (celebdf best accuracy). ffpp_vox는 분리 실패 → 0.5 유지.
THRESHOLDS = {
    "celebdf": 0.19,
    "ffpp_vox": 0.5,
}

OUT_DIR = NOTEBOOK_DIR / "output" / "pwcnet-cm"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS_TEAM = ["fake", "real"]

for k, p in LOCAL_FILES.items():
    if not p.is_file():
        raise FileNotFoundError(f"{k}: file not found -> {p}")
    print(f"{k}: {p}")
print("OK: both JSON files found")
print("THRESHOLDS:", THRESHOLDS)


In [ ]:
def _item_score(item: dict) -> float | None:
    val = item.get("fake_score", item.get("motion_anomaly_score"))
    if val is None:
        return None
    return float(val)


def pred_labels(items: list[dict], threshold: float) -> list[str]:
    out: list[str] = []
    for item in items:
        score = _item_score(item)
        if score is None:
            out.append("real")
        else:
            out.append("fake" if score >= threshold else "real")
    return out


def load_items(path: Path, profile_key: str) -> tuple[list[dict], float]:
    data = json.loads(path.read_text(encoding="utf-8"))
    assert data["model"] == "pwcnet", f"expected pwcnet, got {data.get('model')} in {path.name}"
    threshold = THRESHOLDS[profile_key]
    print(
        f"{data['model']} / {data['profile']} / n={data['count']}  thr={threshold}  ({path.name})"
    )
    items = [
        i for i in data.get("items", [])
        if i.get("status", "ok") == "ok" and i.get("ground_truth_label") in ("fake", "real")
    ]
    return items, threshold


items_ffpp, thr_ffpp = load_items(LOCAL_FILES["ffpp_vox"], "ffpp_vox")
items_celeb, thr_celeb = load_items(LOCAL_FILES["celebdf"], "celebdf")
items_all = items_ffpp + items_celeb

print(f"total: {len(items_all)}")

y_true = [it["ground_truth_label"] for it in items_all]
y_pred = pred_labels(items_ffpp, thr_ffpp) + pred_labels(items_celeb, thr_celeb)


In [ ]:
def plot_team_cm(y_true, y_pred, title: str, out_path: Path | None = None):
    cm = confusion_matrix(y_true, y_pred, labels=LABELS_TEAM)
    tp, fn, fp, tn = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]

    annot = np.array([[f"{tp}\nTP", f"{fn}\nFN"], [f"{fp}\nFP", f"{tn}\nTN"]])

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=annot,
        fmt="",
        cmap="Blues",
        cbar=True,
        xticklabels=["Pred 가짜 (Pos)", "Pred 진짜 (Neg)"],
        yticklabels=["Actual 가짜 (Pos)", "Actual 진짜 (Neg)"],
        ax=ax,
    )
    ax.set_xlabel("Predicted Values")
    ax.set_ylabel("Actual Values")
    ax.set_title(title)
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        print(f"saved {out_path}")
    plt.show()
    return cm


thr_note = f"celebdf={THRESHOLDS['celebdf']}, ffpp_vox={THRESHOLDS['ffpp_vox']}"
cm200 = plot_team_cm(
    y_true,
    y_pred,
    title=f"PWC-Net — Confusion Matrix (n={len(items_all)}, {thr_note})",
    out_path=OUT_DIR / "cm_pwcnet_200_combined.png",
)
print("CM (rows=fake,real):\n", cm200)
print(classification_report(y_true, y_pred, labels=LABELS_TEAM, digits=3))


In [ ]:
# (선택) 프로필별 2장 — ffpp_vox vs celebdf 비교용
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, name, items, thr in [
    (axes[0], "ffpp_vox (100)", items_ffpp, thr_ffpp),
    (axes[1], "celebdf (100)", items_celeb, thr_celeb),
]:
    yt = [it["ground_truth_label"] for it in items]
    yp = pred_labels(items, thr)
    cm = confusion_matrix(yt, yp, labels=LABELS_TEAM)
    annot = np.array(
        [[f"{cm[0,0]}\nTP", f"{cm[0,1]}\nFN"], [f"{cm[1,0]}\nFP", f"{cm[1,1]}\nTN"]]
    )
    sns.heatmap(
        cm,
        annot=annot,
        fmt="",
        cmap="Blues",
        xticklabels=["Pred 가짜", "Pred 진짜"],
        yticklabels=["Actual 가짜", "Actual 진짜"],
        ax=ax,
        cbar=False,
    )
    ax.set_title(f"PWC-Net / {name} (thr={thr})")
fig.suptitle("PWC-Net — profile별 Confusion Matrix")
fig.tight_layout()
split_path = OUT_DIR / "cm_pwcnet_200_by_profile.png"
fig.savefig(split_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"saved {split_path}")


## ROC Curve (200개)

- **Positive = fake(1)**, score = `fake_score` / `motion_anomaly_score`
- **AUC**: 1에 가까울수록 좋음. 점선 = 랜덤(0.5). ● = 프로필 threshold 지점.


In [ ]:
from sklearn.metrics import roc_curve, auc
from IPython.display import display

SCORE_FIELD = "fake_score"
MODEL_LABEL = "PWC-Net"


def items_to_roc_arrays(items) -> tuple[np.ndarray, np.ndarray]:
    y_list: list[int] = []
    s_list: list[float] = []
    skipped = 0
    for it in items:
        score = _item_score(it)
        if score is None:
            skipped += 1
            continue
        y_list.append(1 if it["ground_truth_label"] == "fake" else 0)
        s_list.append(score)
    if skipped:
        print(f"ROC: skipped {skipped} item(s) with no score")
    return np.array(y_list), np.array(s_list)


def plot_roc_single(items, title: str, out_path: Path | None = None, thr: float = 0.5) -> float:
    y, s = items_to_roc_arrays(items)
    fpr, tpr, thresholds = roc_curve(y, s, pos_label=1)
    roc_auc = auc(fpr, tpr)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, lw=2, label=f"ROC (AUC = {roc_auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random (AUC = 0.5)")
    if len(thresholds) > 0:
        idx = int(np.argmin(np.abs(thresholds - thr)))
        ax.scatter(fpr[idx], tpr[idx], s=80, zorder=5, label=f"thr={thr}")
    ax.set_xlabel("FPR (False Positive Rate, 오탐률)")
    ax.set_ylabel("TPR (True Positive Rate, 재현도)")
    ax.set_title(title)
    ax.legend(loc="lower right")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        print(f"saved {out_path}")
    plt.show()
    return roc_auc


n_all = len(items_all)
auc_all = plot_roc_single(
    items_all,
    f"{MODEL_LABEL} — ROC Curve (n={n_all}, score={SCORE_FIELD})",
    OUT_DIR / f"roc_pwcnet_{n_all}_combined.png",
    thr=0.5,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
auc_rows = []
for ax, key, items, thr in [
    (axes[0], "ffpp_vox (100)", items_ffpp, thr_ffpp),
    (axes[1], "celebdf (100)", items_celeb, thr_celeb),
]:
    y, s = items_to_roc_arrays(items)
    fpr, tpr, thresholds = roc_curve(y, s, pos_label=1)
    a = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, label=f"AUC = {a:.3f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    if len(thresholds) > 0:
        idx = int(np.argmin(np.abs(thresholds - thr)))
        ax.scatter(fpr[idx], tpr[idx], s=60, zorder=5, label=f"thr={thr}")
    ax.set_xlabel("FPR (오탐률)")
    ax.set_ylabel("TPR (재현도)")
    ax.set_title(f"{MODEL_LABEL} / {key}")
    ax.legend(loc="lower right")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    auc_rows.append({"구분": key.split()[0], "AUC": f"{a:.4f}", "n": len(items)})

fig.suptitle(f"{MODEL_LABEL} — profile별 ROC (score={SCORE_FIELD})")
fig.tight_layout()
split_roc = OUT_DIR / f"roc_pwcnet_{n_all}_by_profile.png"
fig.savefig(split_roc, dpi=150, bbox_inches="tight")
plt.show()
print(f"saved {split_roc}")

display(
    pd.DataFrame(
        [{"구분": f"{n_all} combined", "AUC": f"{auc_all:.4f}", "n": n_all}] + auc_rows
    )
)


## 분류 지표 추출 (Accuracy · Precision · Recall · F1)

Confusion Matrix에서 TP/FN/FP/TN을 읽어 Xception/EfficientNet 노트북과 동일 공식으로 계산합니다.

**Positive = 가짜(`fake`)** 기준:

| 지표 | 공식 |
|------|------|
| Accuracy(정확도) | (TP + TN) / (TP + FN + FP + TN) |
| Precision(정밀도) | TP / (TP + FP) |
| Recall(재현도) | TP / (TP + FN) |
| F1-Score(조화평균) | 2 × Precision(정밀도) × Recall(재현도) / (Precision(정밀도) + Recall(재현도)) |


In [ ]:
from IPython.display import Markdown, display


def metrics_from_cm(cm: np.ndarray) -> dict:
    tp, fn, fp, tn = int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1])
    total = tp + fn + fp + tn
    accuracy = (tp + tn) / total
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {
        "TP": tp, "FN": fn, "FP": fp, "TN": tn, "n": total,
        "accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1,
        "acc_num": tp + tn,
        "acc_den": total,
        "prec_num": tp,
        "prec_den": tp + fp,
        "rec_num": tp,
        "rec_den": tp + fn,
        "f1_num": 2 * tp,
        "f1_den": 2 * tp + fp + fn,
    }


def metrics_table(m: dict) -> pd.DataFrame:
    # 지표별 분자·분모·값 표
    rows = [
        {
            "지표": "Accuracy(정확도)",
            "분자": f"TP + TN = {m['TP']} + {m['TN']} = {m['acc_num']}",
            "분모": f"TP + FN + FP + TN = {m['n']}",
            "수식": f"{m['acc_num']} / {m['acc_den']}",
            "값": f"{m['accuracy']:.4f}",
            "백분율": f"{m['accuracy'] * 100:.2f}%",
        },
        {
            "지표": "Precision(정밀도)",
            "분자": f"TP = {m['prec_num']}",
            "분모": f"TP + FP = {m['TP']} + {m['FP']} = {m['prec_den']}",
            "수식": f"{m['prec_num']} / {m['prec_den']}",
            "값": f"{m['precision']:.4f}",
            "백분율": f"{m['precision'] * 100:.2f}%",
        },
        {
            "지표": "Recall(재현도)",
            "분자": f"TP = {m['rec_num']}",
            "분모": f"TP + FN = {m['TP']} + {m['FN']} = {m['rec_den']}",
            "수식": f"{m['rec_num']} / {m['rec_den']}",
            "값": f"{m['recall']:.4f}",
            "백분율": f"{m['recall'] * 100:.2f}%",
        },
        {
            "지표": "F1-Score(조화평균)",
            "분자": f"2 × TP = 2 × {m['TP']} = {m['f1_num']}",
            "분모": f"2×TP + FP + FN = {m['f1_den']}",
            "수식": f"{m['f1_num']} / {m['f1_den']}",
            "값": f"{m['f1']:.4f}",
            "백분율": f"{m['f1'] * 100:.2f}%",
        },
    ]
    return pd.DataFrame(rows)


def display_metrics_block(label: str, yt, yp) -> dict:
    cm = confusion_matrix(yt, yp, labels=LABELS_TEAM)
    m = metrics_from_cm(cm)
    tp, fn, fp, tn = m["TP"], m["FN"], m["FP"], m["TN"]

    display(Markdown(f"### {label}  \n**n = {m['n']}** · Positive = fake (가짜)"))

    display(Markdown(
        f''' 
| | Pred fake | Pred real |
|---|---:|---:|
| **Actual fake** | **TP = {tp}** | FN = {fn} |
| **Actual real** | FP = {fp} | **TN = {tn}** |
'''
    ))

    display(metrics_table(m))

    display(Markdown(
        f'''
**LaTeX 형태**

$$
\\text{{Accuracy(정확도)}} = \\frac{{TP + TN}}{{TP + FN + FP + TN}}
= \\frac{{{tp} + {tn}}}{{{m['n']}}}
= \\frac{{{m['acc_num']}}}{{{m['acc_den']}}}
= {m['accuracy']:.4f}
$$

$$
\\text{{Precision(정밀도)}} = \\frac{{TP}}{{TP + FP}}
= \\frac{{{tp}}}{{{tp} + {fp}}}
= \\frac{{{m['prec_num']}}}{{{m['prec_den']}}}
= {m['precision']:.4f}
$$

$$
\\text{{Recall(재현도)}} = \\frac{{TP}}{{TP + FN}}
= \\frac{{{tp}}}{{{tp} + {fn}}}
= \\frac{{{m['rec_num']}}}{{{m['rec_den']}}}
= {m['recall']:.4f}
$$

$$
\\text{{F1-Score(조화평균)}} = \\frac{{2 \\times \\text{{Precision(정밀도)}} \\times \\text{{Recall(재현도)}}}}{{\\text{{Precision(정밀도)}} + \\text{{Recall(재현도)}}}}
= \\frac{{2 \\times TP}}{{2 \\times TP + FP + FN}}
= \\frac{{{m['f1_num']}}}{{{m['f1_den']}}}
= {m['f1']:.4f}
$$
'''
    ))
    return m


yt_ffpp = [it["ground_truth_label"] for it in items_ffpp]
yp_ffpp = pred_labels(items_ffpp, thr_ffpp)
yt_celeb = [it["ground_truth_label"] for it in items_celeb]
yp_celeb = pred_labels(items_celeb, thr_celeb)

m_all = display_metrics_block("PWC-Net 200 (ffpp_vox + celebdf)", y_true, y_pred)
m_ffpp = display_metrics_block(f"PWC-Net · ffpp_vox (thr={thr_ffpp})", yt_ffpp, yp_ffpp)
m_celeb = display_metrics_block(f"PWC-Net · celebdf (thr={thr_celeb})", yt_celeb, yp_celeb)

summary = pd.DataFrame([
    {
        "구분": "200 combined",
        "threshold": f"ffpp={thr_ffpp}, celeb={thr_celeb}",
        "TP": m_all["TP"], "FN": m_all["FN"], "FP": m_all["FP"], "TN": m_all["TN"],
        "Accuracy(정확도)": f"{m_all['acc_num']}/{m_all['acc_den']} = {m_all['accuracy']:.3f}",
        "Precision(정밀도)": f"{m_all['prec_num']}/{m_all['prec_den']} = {m_all['precision']:.3f}",
        "Recall(재현도)": f"{m_all['rec_num']}/{m_all['rec_den']} = {m_all['recall']:.3f}",
        "F1-Score(조화평균)": f"{m_all['f1_num']}/{m_all['f1_den']} = {m_all['f1']:.3f}",
    },
    {
        "구분": "ffpp_vox",
        "threshold": str(thr_ffpp),
        "TP": m_ffpp["TP"], "FN": m_ffpp["FN"], "FP": m_ffpp["FP"], "TN": m_ffpp["TN"],
        "Accuracy(정확도)": f"{m_ffpp['acc_num']}/{m_ffpp['acc_den']} = {m_ffpp['accuracy']:.3f}",
        "Precision(정밀도)": f"{m_ffpp['prec_num']}/{m_ffpp['prec_den']} = {m_ffpp['precision']:.3f}",
        "Recall(재현도)": f"{m_ffpp['rec_num']}/{m_ffpp['rec_den']} = {m_ffpp['recall']:.3f}",
        "F1-Score(조화평균)": f"{m_ffpp['f1_num']}/{m_ffpp['f1_den']} = {m_ffpp['f1']:.3f}",
    },
    {
        "구분": "celebdf",
        "threshold": str(thr_celeb),
        "TP": m_celeb["TP"], "FN": m_celeb["FN"], "FP": m_celeb["FP"], "TN": m_celeb["TN"],
        "Accuracy(정확도)": f"{m_celeb['acc_num']}/{m_celeb['acc_den']} = {m_celeb['accuracy']:.3f}",
        "Precision(정밀도)": f"{m_celeb['prec_num']}/{m_celeb['prec_den']} = {m_celeb['precision']:.3f}",
        "Recall(재현도)": f"{m_celeb['rec_num']}/{m_celeb['rec_den']} = {m_celeb['recall']:.3f}",
        "F1-Score(조화평균)": f"{m_celeb['f1_num']}/{m_celeb['f1_den']} = {m_celeb['f1']:.3f}",
    },
])

display(Markdown("### 전체 요약"))
display(summary)
